# Flash Flash Revolution Gold Layer Features

This notebook creates the **gold layer feature table** by joining all silver layer features into a unified feature vector for TCN difficulty prediction.

## Incremental Processing

**Change Detection:**
- Monitors `swf_version` from `bronze__songlist` to detect song updates
- Only processes songs that are new or have changed since last run
- Uses DELETE + INSERT pattern for incremental updates
- Set `process_all = True` to force full refresh

**Dependencies:**
- Silver layer tables: vertical-density, horizontal-density, chart-features
- Playlist and reliability scores for metadata enrichment
- All silver tables must be up-to-date before running gold layer

---

## Feature Vector Architecture

### Note-Level Features (vary by note)

1. **`vertical_density`**: Same-orientation note speed (1/time_delta)
   - Measures how fast notes appear on the same arrow
   - Source: `acubed.ffr.silver__vertical-density`

2. **`horizontal_density`**: Temporal clustering across all arrows
   - Measures how clustered notes are in time (weighted sum approach)
   - Source: `acubed.ffr.silver__horizontal-density`

3. **`position_in_song`**: Note position within song [0, 1]
   - Computed as: note_id / total_notes
   - Enables TCN to learn position-dependent difficulty patterns

### Song-Level Features (constant per song)

4. **`song_length_log_normalized`**: Log-normalized song length [0, ~1.25]
   - 95% of songs in [0, 1], top 5% exceed 1.0
   - Normalized using P95 reference point
   - Source: `acubed.ffr.silver__chart-features`

5. **`difficulty`**: Official difficulty rating [1-120]
   - Source: `acubed.ffr.silver__playlist`
   - Used as target variable for prediction

### Game Mechanics Placeholders

6. **`hold`**: Constant **0**
   - **FFR has NO hold notes** (unlike DDR/ITG/Stepmania)
   - Preserves architecture for cross-game compatibility

7. **`mine`**: Constant **1**
   - **Conceptual interpretation**: Omnipresent mine threat
   - In FFR, pressing the WRONG arrow at ANY time breaks your combo
   - Every moment between notes is a potential mistake

### Sample Weight for Training

8. **`sample_weight`**: Smoothed reliability score [~0.19 - 0.33]
   - **Use as sample weight during model training** - higher values = more confidence in the difficulty rating
   - Computed from contested difficulty assessments with ordinal smoothing
   - Source: `acubed.ffr.silver__reliability-scores` (avg_reliability column)

---

## Output Table

**Table:** `acubed.ffr.gold__features`

**Schema:** Note-level features with sample weights
- `song_id`, `note_id`, `orientation`
- `vertical_density`, `horizontal_density`, `position_in_song`
- `song_length_log_normalized`, `hold`, `mine`
- `difficulty`, `sample_weight`
- `swf_version`

**Granularity:** One row per decomposed note

---

## Source Tables

**Silver Layer:**
- `acubed.ffr.silver__vertical-density` (note-level vertical speed)
- `acubed.ffr.silver__horizontal-density` (note-level temporal clustering)
- `acubed.ffr.silver__chart-features` (song-level normalized length)
- `acubed.ffr.silver__playlist` (difficulty ratings)
- `acubed.ffr.silver__reliability-scores` (avg_reliability for sample weighting)

**Bronze Layer:**
- `acubed.ffr.bronze__songlist` (swf_version for change detection)

In [0]:
from pyspark.sql.functions import col, lit

print("✓ Imports loaded successfully")

In [0]:
# Configuration: Automatic Processing Mode
# Automatically determines whether to do full refresh or incremental processing
# - Full refresh (process_all=True): When table doesn't exist (first run)
# - Incremental (process_all=False): When table exists (subsequent runs)

gold_table = "acubed.ffr.`gold__features`"
process_all = not spark.catalog.tableExists(gold_table)

if process_all:
    print(f"ℹ Auto-detected: Full refresh mode (table does not exist)")
else:
    print(f"ℹ Auto-detected: Incremental mode (table exists)")

print(f"✓ Configuration loaded: process_all = {process_all}")

In [0]:
# Identify songs that need to be processed (new or updated)
# Uses swf_version from bronze__songlist to detect changes

gold_table = "acubed.ffr.`gold__features`"

if spark.catalog.tableExists(gold_table) and not process_all:
    # Check if gold table has swf_version column (migration check)
    gold_columns = [field.name for field in spark.table(gold_table).schema.fields]
    
    if "swf_version" not in gold_columns:
        print("ℹ Gold table missing swf_version column - forcing full refresh to migrate schema")
        changed_song_ids = None
    else:
        # Gold table exists with swf_version - check for new songs and updated songs
        bronze_songlist = spark.table("acubed.ffr.bronze__songlist").select(
            col("id").alias("song_id"),
            col("swf_version").alias("bronze_swf_version")
        )
        
        gold_features = spark.table(gold_table).select(
            col("song_id").alias("gold_song_id"),
            col("swf_version").alias("gold_swf_version")
        ).distinct()
        
        # Left join to find new songs and updated songs
        changed_songs = bronze_songlist.join(
            gold_features,
            bronze_songlist.song_id == gold_features.gold_song_id,
            "left"
        ).filter(
            # New songs (not in gold) OR updated songs (different swf_version)
            col("gold_song_id").isNull() |
            (col("bronze_swf_version") != col("gold_swf_version"))
        )
        
        changed_song_ids = changed_songs.select(col("song_id")).distinct()
        num_changed = changed_song_ids.count()
        
        if num_changed > 0:
            print(f"ℹ Incremental mode: Found {num_changed} changed songs")
        else:
            print("ℹ No changes detected - gold table is up-to-date")
            changed_song_ids = None
else:
    # Force full refresh or table doesn't exist
    if process_all:
        print("ℹ Force full refresh mode enabled (process_all=True)")
    else:
        print("ℹ Gold table does not exist - will create from scratch")
    
    changed_song_ids = None
    print("✓ Will process all songs")

In [0]:
# Guard: Skip if no songs to process
if not process_all and changed_song_ids is None:
    print("ℹ No songs to process - skipping feature vector creation")
    print("✓ Notebook will complete successfully (idempotent run)")
    
    # Create empty DataFrame with correct schema for downstream compatibility
    df_features = spark.createDataFrame([], schema="""
        song_id bigint,
        note_id int,
        orientation string,
        vertical_density double,
        horizontal_density double,
        position_in_song double,
        song_length_log_normalized double,
        hold int,
        mine int,
        swf_version string
    """)
else:
    # Join all silver layer features and create feature vector
    df_features = (
        spark.table("acubed.ffr.`silver__vertical-density`").alias("v")
        .join(
            spark.table("acubed.ffr.`silver__horizontal-density`").alias("h"),
            ["song_id", "note_id", "orientation"]
        )
        .join(
            spark.table("acubed.ffr.`silver__chart-features`").alias("s"),
            "song_id"
        )
        .withColumn(
            "position_in_song",  # Compute position on-the-fly
            col("note_id") / col("s.note_count")
        )
        .withColumn(
            "hold",  # No holds in FFR (game mechanic)
            lit(0)
        )
        .withColumn(
            "mine",  # Omnipresent mine threat (infinite mines between notes)
            lit(1)
        )
        .select(
            "song_id", "note_id", "orientation",
            col("v.vertical_density"),          # Feature 1: Same-arrow speed
            col("h.horizontal_density"),        # Feature 2: Cross-arrow clustering
            "position_in_song",                 # Feature 3: [0, 1] position
            col("s.song_length_log_normalized"), # Feature 4: ~[0, 1.25] length
            "hold",                              # Feature 5: Constant 0 (no holds)
            "mine",                              # Feature 6: Constant 1 (omnipresent)
            col("v.swf_version")                # Version tracking for incremental updates
        )
    )

    # Filter to changed songs if doing incremental processing
    if not process_all and changed_song_ids is not None:
        df_features = df_features.join(changed_song_ids, "song_id", "inner")
        print(f"ℹ Incremental mode: Built features for {df_features.count():,} notes from changed songs")
    else:
        print(f"✓ Full refresh mode: Building features for all notes...")

    print("\nFeature columns:")
    print("  1. vertical_density (same-arrow note speed)")
    print("  2. horizontal_density (cross-arrow temporal clustering)")
    print("  3. position_in_song (note position [0,1])")
    print("  4. song_length_log_normalized (song length [0,~1.25])")
    print("  5. hold (constant 0 - no holds in FFR)")
    print("  6. mine (constant 1 - omnipresent mine threat)")
    print("  + swf_version (for incremental update tracking)")

In [0]:
# Add reliability scores to features by joining through playlist
print("\n" + "="*60)
print("ADDING RELIABILITY SCORES")
print("="*60)

# Join features -> playlist (get difficulty) -> reliability scores
df_playlist = spark.table("acubed.ffr.silver__playlist").select("song_id", "difficulty")
df_reliability = spark.table("acubed.ffr.`silver__reliability-scores`").select("new_difficulty", col("avg_reliability").alias("sample_weight"))

df_features = df_features.join(
    df_playlist,
    "song_id",
    "left"
).join(
    df_reliability,
    col("difficulty") == df_reliability.new_difficulty,  # Use col() to reference newly added column
    "left"
).drop("new_difficulty")

print("✓ Added: difficulty, sample_weight")
print("\nUse sample_weight for training - higher values = more confidence")
print("="*60)

In [0]:
display(df_features)

In [0]:
from delta.tables import DeltaTable

# Save to Delta table using DELETE + INSERT pattern for incremental updates
gold_table = "acubed.ffr.`gold__features`"

print("Saving to gold layer...")
print("=" * 60)

# Capture counts BEFORE operations
rows_to_insert = df_features.count()

if rows_to_insert == 0:
    print("ℹ No rows to insert - skipping table update (idempotent run)")
else:
    if spark.catalog.tableExists(gold_table) and not process_all:
        # Table exists and incremental mode - use DELETE + INSERT
        delta_table = DeltaTable.forName(spark, gold_table)
        
        # Get list of song_ids being updated
        updated_song_ids = df_features.select("song_id").distinct()
        
        # Delete existing rows for these songs
        song_ids_to_delete = [row.song_id for row in updated_song_ids.collect()]
        
        if len(song_ids_to_delete) > 0:
            # Build condition for DELETE
            delete_condition = col("song_id").isin(song_ids_to_delete)
            delta_table.delete(delete_condition)
            print(f"✓ Deleted existing rows for {len(song_ids_to_delete)} songs")
        
        # Insert new rows
        df_features.write.format("delta").mode("append").saveAsTable(gold_table)
        print(f"✓ Inserted {rows_to_insert:,} new rows")
        
        # Get final count
        final_count = spark.table(gold_table).count()
        print(f"✓ Updated {gold_table}")
        print(f"  Total rows in table: {final_count:,}")
    else:
        # Table doesn't exist or full refresh mode - overwrite with schema migration
        df_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table)
        
        if process_all:
            print(f"✓ Full refresh: Recreated {gold_table} with {rows_to_insert:,} rows")
        else:
            print(f"✓ Created table {gold_table} with {rows_to_insert:,} rows")

print("\n" + "=" * 60)
print("✓ Gold features table ready for TCN model training!")
print("=" * 60)